In [1]:
import pandas as pd
import numpy as np
import time
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score, mean_absolute_percentage_error
from sklearn.metrics import explained_variance_score, max_error, median_absolute_error
import lightgbm as lgb
import optuna
from optuna.samplers import TPESampler
import joblib
import matplotlib.pyplot as plt
import seaborn as sns

# Load the data
data = pd.read_csv('preprocessed.csv')

# Define features and target
features = ['current_stop_name', 'next_stop_name', 'day_of_week', 'is_holiday', 
            'is_peak_hour', 'weather_condition', 'passenger_count', 'current_speed', 
            'distance_to_next_stop', 'current_lat', 'current_lon']
target = 'eta_minutes'

X = data[features]
y = data[target]

# Convert boolean columns to int
X['is_holiday'] = X['is_holiday'].astype(int)
X['is_peak_hour'] = X['is_peak_hour'].astype(int)

# Split data into train and test sets
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Enhanced function to evaluate model with additional metrics
def evaluate_model(model, X_train, y_train, X_test, y_test):
    # Training set predictions
    y_train_pred = model.predict(X_train)
    
    # Test set predictions
    y_test_pred = model.predict(X_test)
    
    # Calculate metrics for test set
    mae = mean_absolute_error(y_test, y_test_pred)
    mse = mean_squared_error(y_test, y_test_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_test_pred)
    
    # Additional metrics for test set
    explained_var = explained_variance_score(y_test, y_test_pred)
    max_err = max_error(y_test, y_test_pred)
    median_ae = median_absolute_error(y_test, y_test_pred)
    
    # Calculate MAPE (handling zero values to avoid division by zero)   
    mape = calculate_safe_mape(y_test, y_test_pred)  # Convert to percentage
    
    # Calculate metrics for training set to check for overfitting
    train_mae = mean_absolute_error(y_train, y_train_pred)
    train_mse = mean_squared_error(y_train, y_train_pred)
    train_rmse = np.sqrt(train_mse)
    train_r2 = r2_score(y_train, y_train_pred)
    
    # Print results
    print("\n===== Test Set Metrics =====")
    print(f"MAE: {mae:.4f}")
    print(f"MSE: {mse:.4f}")
    print(f"RMSE: {rmse:.4f}")
    print(f"MAPE: {mape:.2f}%" if not np.isnan(mape) else "MAPE: Cannot calculate (zero values in actual data)")
    print(f"R² Score: {r2:.4f}")
    print(f"Explained Variance: {explained_var:.4f}")
    print(f"Max Error: {max_err:.4f}")
    print(f"Median Absolute Error: {median_ae:.4f}")
    
    print("\n===== Training Set Metrics =====")
    print(f"Training MAE: {train_mae:.4f}")
    print(f"Training MSE: {train_mse:.4f}")
    print(f"Training RMSE: {train_rmse:.4f}")
    print(f"Training R² Score: {train_r2:.4f}")
    
    # Calculate overfitting ratio (train error / test error)
    overfitting_ratio = train_rmse / rmse
    print(f"\nOverfitting Ratio (train RMSE / test RMSE): {overfitting_ratio:.4f}")
    print(f"Note: Ratio close to 1.0 indicates less overfitting")
    
    # Return all metrics as a dictionary
    return {
        'Test_MAE': mae, 
        'Test_MSE': mse, 
        'Test_RMSE': rmse,
        'Test_MAPE': mape,
        'Test_R2': r2,
        'Test_Explained_Variance': explained_var,
        'Test_Max_Error': max_err,
        'Test_Median_AE': median_ae,
        'Train_MAE': train_mae,
        'Train_MSE': train_mse,
        'Train_RMSE': train_rmse,
        'Train_R2': train_r2,
        'Overfitting_Ratio': overfitting_ratio
    }

# Calculate cross-validation scores
def calculate_cv_scores(model, X, y, cv=5):
    print(f"\n===== {cv}-Fold Cross Validation Scores =====")
    
    # Calculate CV scores for multiple metrics
    cv_rmse = np.sqrt(-cross_val_score(model, X, y, scoring='neg_mean_squared_error', cv=cv, n_jobs=-1))
    cv_mae = -cross_val_score(model, X, y, scoring='neg_mean_absolute_error', cv=cv, n_jobs=-1)
    cv_r2 = cross_val_score(model, X, y, scoring='r2', cv=cv, n_jobs=-1)
    
    # Print results
    print(f"CV RMSE: {cv_rmse.mean():.4f} ± {cv_rmse.std():.4f}")
    print(f"CV MAE: {cv_mae.mean():.4f} ± {cv_mae.std():.4f}")
    print(f"CV R²: {cv_r2.mean():.4f} ± {cv_r2.std():.4f}")
    
    return {
        'CV_RMSE_mean': cv_rmse.mean(),
        'CV_RMSE_std': cv_rmse.std(),
        'CV_MAE_mean': cv_mae.mean(),
        'CV_MAE_std': cv_mae.std(),
        'CV_R2_mean': cv_r2.mean(),
        'CV_R2_std': cv_r2.std()
    }

def calculate_safe_mape(y_true, y_pred, epsilon=1e-10):
    """
    Calculate MAPE with protection against division by zero.
    
    Args:
        y_true: Actual values
        y_pred: Predicted values
        epsilon: Small constant to avoid division by zero
        
    Returns:
        MAPE as percentage
    """
    # Remove pairs where actual value is zero or extremely small
    mask = np.abs(y_true) > epsilon
    if mask.sum() == 0:  # If no valid samples remain
        return np.nan
        
    # Calculate MAPE only on valid samples
    y_true_safe = y_true[mask]
    y_pred_safe = y_pred[mask]
    
    # Calculate absolute percentage errors for each point
    abs_percentage_errors = np.abs((y_true_safe - y_pred_safe) / y_true_safe) * 100
    
    # Return mean
    return np.mean(abs_percentage_errors)

# Plot residuals
def plot_residuals(model, X_test, y_test, title):
    y_pred = model.predict(X_test)
    residuals = y_test - y_pred
    
    plt.figure(figsize=(12, 6))
    
    # Plot 1: Residuals vs Predicted
    plt.subplot(1, 2, 1)
    plt.scatter(y_pred, residuals, alpha=0.5)
    plt.axhline(y=0, color='r', linestyle='-')
    plt.xlabel('Predicted ETA (minutes)')
    plt.ylabel('Residuals')
    plt.title(f'Residuals vs Predicted: {title}')
    
    # Plot 2: Residuals Distribution
    plt.subplot(1, 2, 2)
    sns.histplot(residuals, kde=True)
    plt.axvline(x=0, color='r', linestyle='-')
    plt.xlabel('Residual Value')
    plt.ylabel('Frequency')
    plt.title(f'Residuals Distribution: {title}')
    
    plt.tight_layout()
    plt.savefig(f'residuals_{title.lower().replace(" ", "_")}.png')
    plt.close()
    
    # Calculate residual statistics
    res_mean = residuals.mean()
    res_std = residuals.std()
    
    print(f"\n===== Residual Analysis for {title} =====")
    print(f"Mean of Residuals: {res_mean:.4f}")
    print(f"Std Dev of Residuals: {res_std:.4f}")
    
    return {
        'Residual_Mean': res_mean,
        'Residual_StdDev': res_std
    }

# Plot actual vs predicted
def plot_actual_vs_predicted(model, X_test, y_test, title):
    y_pred = model.predict(X_test)
    
    plt.figure(figsize=(10, 6))
    plt.scatter(y_test, y_pred, alpha=0.5)
    
    # Add perfect prediction line
    min_val = min(min(y_test), min(y_pred))
    max_val = max(max(y_test), max(y_pred))
    plt.plot([min_val, max_val], [min_val, max_val], 'r--')
    
    plt.xlabel('Actual ETA (minutes)')
    plt.ylabel('Predicted ETA (minutes)')
    plt.title(f'Actual vs Predicted: {title}')
    plt.tight_layout()
    plt.savefig(f'actual_vs_predicted_{title.lower().replace(" ", "_")}.png')
    plt.close()

# Plot prediction error distribution
def plot_error_distribution(model, X_test, y_test, title):
    y_pred = model.predict(X_test)
    abs_errors = np.abs(y_test - y_pred)
    
    plt.figure(figsize=(10, 6))
    sns.histplot(abs_errors, kde=True, bins=30)
    plt.axvline(abs_errors.mean(), color='r', linestyle='--', label=f'Mean: {abs_errors.mean():.2f}')
    plt.axvline(np.median(abs_errors), color='g', linestyle='--', label=f'Median: {np.median(abs_errors):.2f}')
    
    # Calculate percentiles
    p90 = np.percentile(abs_errors, 90)
    p95 = np.percentile(abs_errors, 95)
    p99 = np.percentile(abs_errors, 99)
    
    plt.axvline(p90, color='orange', linestyle=':', label=f'90th %: {p90:.2f}')
    plt.axvline(p95, color='purple', linestyle=':', label=f'95th %: {p95:.2f}')
    
    plt.xlabel('Absolute Prediction Error (minutes)')
    plt.ylabel('Frequency')
    plt.title(f'Prediction Error Distribution: {title}')
    plt.legend()
    plt.tight_layout()
    plt.savefig(f'error_distribution_{title.lower().replace(" ", "_")}.png')
    plt.close()
    
    print(f"\n===== Error Distribution for {title} =====")
    print(f"90% of errors are below: {p90:.4f} minutes")
    print(f"95% of errors are below: {p95:.4f} minutes")
    print(f"99% of errors are below: {p99:.4f} minutes")
    
    return {
        'Error_P90': p90,
        'Error_P95': p95,
        'Error_P99': p99
    }

# Baseline LightGBM model
print("\n" + "="*50)
print("Training baseline LightGBM model...")
print("="*50)

# Measure training time for baseline model
baseline_start_time = time.time()
baseline_model = lgb.LGBMRegressor(random_state=42)
baseline_model.fit(X_train, y_train)
baseline_training_time = time.time() - baseline_start_time

print(f"\nBaseline Model Training Time: {baseline_training_time:.2f} seconds")

print("\nBaseline Model Performance:")
baseline_metrics = evaluate_model(baseline_model, X_train, y_train, X_test, y_test)

# Calculate CV scores for baseline model
baseline_cv_metrics = calculate_cv_scores(baseline_model, X, y)

# Plot baseline model evaluation
plot_residuals(baseline_model, X_test, y_test, "Baseline Model")
plot_actual_vs_predicted(baseline_model, X_test, y_test, "Baseline Model")
baseline_error_metrics = plot_error_distribution(baseline_model, X_test, y_test, "Baseline Model")

# Optuna optimization
def objective(trial):
    params = {
        'objective': 'regression',
        'metric': 'rmse',
        'boosting_type': 'gbdt',
        'random_state': 42,
        'verbosity': -1,
        'n_estimators': trial.suggest_int('n_estimators', 50, 500),
        'learning_rate': trial.suggest_float('learning_rate', 0.01, 0.3, log=True),
        'num_leaves': trial.suggest_int('num_leaves', 20, 300),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 100),
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_float('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_float('reg_alpha', 0.0, 10.0),
        'reg_lambda': trial.suggest_float('reg_lambda', 0.0, 10.0),
    }
    
    model = lgb.LGBMRegressor(**params)
    model.fit(X_train, y_train)
    
    y_pred = model.predict(X_test)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    
    return rmse

print("\n" + "="*50)
print("Starting Optuna optimization...")
print("="*50)

optuna_start_time = time.time()
study = optuna.create_study(direction='minimize', sampler=TPESampler(seed=42))
study.optimize(objective, n_trials=50)
optuna_time = time.time() - optuna_start_time

print(f"\nOptuna Optimization Time: {optuna_time:.2f} seconds")
print(f"Best parameters: {study.best_params}")
print(f"Best RMSE: {study.best_value:.4f}")

# Train optimized model
print("\n" + "="*50)
print("Training optimized model with best parameters...")
print("="*50)

# Measure training time for optimized model
optimized_start_time = time.time()
best_params = study.best_params
optimized_model = lgb.LGBMRegressor(**best_params, random_state=42)
optimized_model.fit(X_train, y_train)
optimized_training_time = time.time() - optimized_start_time

print(f"\nOptimized Model Training Time: {optimized_training_time:.2f} seconds")

print("\nOptimized Model Performance:")
optimized_metrics = evaluate_model(optimized_model, X_train, y_train, X_test, y_test)

# Calculate CV scores for optimized model
optimized_cv_metrics = calculate_cv_scores(optimized_model, X, y)

# Plot optimized model evaluation
plot_residuals(optimized_model, X_test, y_test, "Optimized Model")
plot_actual_vs_predicted(optimized_model, X_test, y_test, "Optimized Model")
optimized_error_metrics = plot_error_distribution(optimized_model, X_test, y_test, "Optimized Model")

# Create a visualization of parameter importance using matplotlib instead of plotly
importance_values = optuna.importance.get_param_importances(study)
importance_df = pd.DataFrame(
    {'Parameter': list(importance_values.keys()), 
     'Importance': list(importance_values.values())}
).sort_values('Importance', ascending=False)

plt.figure(figsize=(12, 8))
sns.barplot(x='Importance', y='Parameter', data=importance_df)
plt.title('Parameter Importance in Optuna Optimization')
plt.tight_layout()
plt.savefig('optuna_param_importance.png')
plt.close()
print("\nParameter importance visualization saved as 'optuna_param_importance.png'")

# Compare baseline and optimized models with extended metrics
print("\n" + "="*50)
print("Performance Comparison:")
print("="*50)

# Create a comprehensive comparison DataFrame
comparison_data = []

# Add all metrics to comparison
for metric_name in baseline_metrics.keys():
    if metric_name in optimized_metrics:
        baseline_val = baseline_metrics[metric_name]
        optimized_val = optimized_metrics[metric_name]
        
        if baseline_val != 0:  # Avoid division by zero
            improvement = baseline_val - optimized_val
            improvement_pct = (improvement / abs(baseline_val)) * 100
            
            # Determine if an improvement is "better" based on the metric
            # For most metrics, lower is better (except R2 and Explained Variance)
            better_is_lower = "R2" not in metric_name and "Explained_Variance" not in metric_name
            
            if (better_is_lower and improvement > 0) or (not better_is_lower and improvement < 0):
                status = "↑ Better"
                improvement_pct = abs(improvement_pct)  # Make percentage positive for "better"
            elif abs(improvement) < 0.0001:  # Very small difference
                status = "↔ Similar"
                improvement_pct = 0
            else:
                status = "↓ Worse"
                improvement_pct = -abs(improvement_pct)  # Make percentage negative for "worse"
        else:
            improvement = baseline_val - optimized_val
            improvement_pct = float('nan')
            status = "N/A"
        
        comparison_data.append({
            'Metric': metric_name,
            'Baseline': baseline_val,
            'Optimized': optimized_val,
            'Absolute Difference': improvement,
            'Percentage Change': f"{improvement_pct:.2f}%" if not np.isnan(improvement_pct) else "N/A",
            'Status': status
        })

# Add training time comparison
train_time_improvement = baseline_training_time - optimized_training_time
train_time_pct = (train_time_improvement / baseline_training_time) * 100 if baseline_training_time > 0 else float('nan')

comparison_data.append({
    'Metric': 'Training_Time_Seconds',
    'Baseline': baseline_training_time,
    'Optimized': optimized_training_time,
    'Absolute Difference': train_time_improvement,
    'Percentage Change': f"{train_time_pct:.2f}%" if not np.isnan(train_time_pct) else "N/A",
    'Status': "↑ Better" if train_time_improvement > 0 else ("↔ Similar" if abs(train_time_improvement) < 0.1 else "↓ Worse")
})

# Create a DataFrame and display the comparison
comparison_df = pd.DataFrame(comparison_data)
print("\nDetailed Performance Comparison:")
print(comparison_df)

# Save the comparison to CSV
comparison_df.to_csv('model_comparison.csv', index=False)
print("Comparison saved to 'model_comparison.csv'")

# Save the optimized model
model_filename = 'optimized_lightgbm_eta_predictor.pkl'
joblib.dump(optimized_model, model_filename)
print(f"\nOptimized model saved as {model_filename}")

# Feature importance analysis
importance = pd.DataFrame({
    'feature': features,
    'importance': optimized_model.feature_importances_
}).sort_values('importance', ascending=False)

print("\nFeature Importance:")
print(importance)

# Visualize feature importance
plt.figure(figsize=(12, 8))
sns.barplot(x='importance', y='feature', data=importance)
plt.title('Feature Importance in Optimized Model')
plt.tight_layout()
plt.savefig('feature_importance.png')
plt.close()
print("Feature importance visualization saved as 'feature_importance.png'")

# Save all metrics to a summary file
with open('model_performance_summary.txt', 'w') as f:
    f.write("=" * 80 + "\n")
    f.write("ETA PREDICTION MODEL PERFORMANCE SUMMARY\n")
    f.write("=" * 80 + "\n\n")
    
    f.write("TRAINING INFORMATION:\n")
    f.write(f"Baseline Model Training Time: {baseline_training_time:.2f} seconds\n")
    f.write(f"Optuna Optimization Time: {optuna_time:.2f} seconds\n")
    f.write(f"Optimized Model Training Time: {optimized_training_time:.2f} seconds\n\n")
    
    f.write("BASELINE MODEL METRICS:\n")
    for k, v in baseline_metrics.items():
        f.write(f"{k}: {v:.4f}\n")
    f.write("\n")
    
    f.write("OPTIMIZED MODEL METRICS:\n")
    for k, v in optimized_metrics.items():
        f.write(f"{k}: {v:.4f}\n")
    f.write("\n")
    
    f.write("CROSS-VALIDATION RESULTS:\n")
    f.write("Baseline Model:\n")
    for k, v in baseline_cv_metrics.items():
        f.write(f"{k}: {v:.4f}\n")
    f.write("\nOptimized Model:\n")
    for k, v in optimized_cv_metrics.items():
        f.write(f"{k}: {v:.4f}\n")
    f.write("\n")
    
    f.write("ERROR DISTRIBUTION METRICS:\n")
    f.write("Baseline Model:\n")
    for k, v in baseline_error_metrics.items():
        f.write(f"{k}: {v:.4f}\n")
    f.write("\nOptimized Model:\n")
    for k, v in optimized_error_metrics.items():
        f.write(f"{k}: {v:.4f}\n")
    f.write("\n")
    
    f.write("BEST HYPERPARAMETERS:\n")
    for param, value in study.best_params.items():
        f.write(f"{param}: {value}\n")

print("\nComplete performance summary saved to 'model_performance_summary.txt'")

c:\Users\cheng\AppData\Local\Programs\Python\Python310\lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
C:\Users\cheng\AppData\Local\Temp\ipykernel_8448\1770688278.py:27: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.html#returning-a-view-versus-a-copy
  X['is_holiday'] = X['is_holiday'].astype(int)
C:\Users\cheng\AppData\Local\Temp\ipykernel_8448\1770688278.py:28: SettingWithCopyWarning: 
A value is trying to be set on a copy of a slice from a DataFrame.
Try using .loc[row_indexer,col_indexer] = value instead

See the caveats in the documentation: https://pandas.pydata.org/pandas-docs/stable/user_guide/indexing.


Training baseline LightGBM model...
[LightGBM] [Info] Auto-choosing row-wise multi-threading, the overhead of testing was 0.000965 seconds.
You can set `force_row_wise=true` to remove the overhead.
And if memory is not enough, you can set `force_col_wise=true`.
[LightGBM] [Info] Total Bins 848
[LightGBM] [Info] Number of data points in the train set: 80000, number of used features: 11
[LightGBM] [Info] Start training from score 2.238962

Baseline Model Training Time: 0.16 seconds

Baseline Model Performance:

===== Test Set Metrics =====
MAE: 0.3270
MSE: 0.2857
RMSE: 0.5345
MAPE: 16.05%
R² Score: 0.9657
Explained Variance: 0.9657
Max Error: 7.2591
Median Absolute Error: 0.1803

===== Training Set Metrics =====
Training MAE: 0.3169
Training MSE: 0.2531
Training RMSE: 0.5030
Training R² Score: 0.9697

Overfitting Ratio (train RMSE / test RMSE): 0.9411
Note: Ratio close to 1.0 indicates less overfitting

===== 5-Fold Cross Validation Scores =====
CV RMSE: 0.5348 ± 0.0084
CV MAE: 0.3275 ±

[I 2025-05-04 02:42:05,542] A new study created in memory with name: no-name-93bc3ae0-bba2-42c0-8974-67ab7c547963



===== Error Distribution for Baseline Model =====
90% of errors are below: 0.7932 minutes
95% of errors are below: 1.0357 minutes
99% of errors are below: 2.0065 minutes

Starting Optuna optimization...


[I 2025-05-04 02:42:06,096] Trial 0 finished with value: 0.5446758377085138 and parameters: {'n_estimators': 218, 'learning_rate': 0.2536999076681772, 'num_leaves': 225, 'max_depth': 8, 'min_child_samples': 19, 'subsample': 0.5779972601681014, 'colsample_bytree': 0.5290418060840998, 'reg_alpha': 8.661761457749352, 'reg_lambda': 6.011150117432088}. Best is trial 0 with value: 0.5446758377085138.
[I 2025-05-04 02:42:08,438] Trial 1 finished with value: 0.5717398160309204 and parameters: {'n_estimators': 369, 'learning_rate': 0.010725209743171996, 'num_leaves': 292, 'max_depth': 11, 'min_child_samples': 25, 'subsample': 0.5909124836035503, 'colsample_bytree': 0.5917022549267169, 'reg_alpha': 3.0424224295953772, 'reg_lambda': 5.247564316322379}. Best is trial 0 with value: 0.5446758377085138.
[I 2025-05-04 02:42:08,646] Trial 2 finished with value: 0.5423367043370092 and parameters: {'n_estimators': 244, 'learning_rate': 0.02692655251486473, 'num_leaves': 191, 'max_depth': 4, 'min_child_sa


Optuna Optimization Time: 47.87 seconds
Best parameters: {'n_estimators': 454, 'learning_rate': 0.012615667411260257, 'num_leaves': 265, 'max_depth': 9, 'min_child_samples': 73, 'subsample': 0.5700120160844919, 'colsample_bytree': 0.9938499284681216, 'reg_alpha': 3.4730612665902423, 'reg_lambda': 8.474090483362852}
Best RMSE: 0.5320

Training optimized model with best parameters...

Optimized Model Training Time: 1.52 seconds

Optimized Model Performance:

===== Test Set Metrics =====
MAE: 0.3158
MSE: 0.2830
RMSE: 0.5320
MAPE: 14.94%
R² Score: 0.9660
Explained Variance: 0.9660
Max Error: 7.9305
Median Absolute Error: 0.1573

===== Training Set Metrics =====
Training MAE: 0.3035
Training MSE: 0.2563
Training RMSE: 0.5063
Training R² Score: 0.9693

Overfitting Ratio (train RMSE / test RMSE): 0.9516
Note: Ratio close to 1.0 indicates less overfitting

===== 5-Fold Cross Validation Scores =====
CV RMSE: 0.5320 ± 0.0088
CV MAE: 0.3156 ± 0.0035
CV R²: 0.9661 ± 0.0014

===== Residual Analysi